In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

<a id="linear-regression-house-price-prediction"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>LINEAR REGRESSION FOR HOUSE PRICE PREDICTION</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    In this notebook, we will build different linear regression models for California house price prediction:
    <br><br>
    1. Explore California Housing Dataset
    <br>
    2. Linear regression (with normal equation and iterative optimization)
    <br>
    3. Polynomial regression
    <br>
    4. Regularized regression models - ridge and lasso
    <br><br>
    We will set regularization rate and polynomial degree with hyper-parameter tuning and cross-validation.
    <br><br>
    We will compare different models in terms of their parameter vectors and mean absolute error on train, development, and test sets.
</p>
</div>

<a id="california-housing-dataset"></a>
<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>CALIFORNIA HOUSING DATASET</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    This notebook introduces the California Housing dataset that we will be using for regression demonstrations.
    <br><br>
    We also list down the steps for typical dataset exploration, which can be broadly applied to any dataset.
</p>
</div>


In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
from sklearn.datasets import fetch_california_housing

In [ ]:
california_housing = fetch_california_housing(as_frame=True)

In [ ]:
type(california_housing)

<a id="bunch-object-attributes"></a>
<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>BUNCH OBJECT ATTRIBUTES</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    The <code>bunch</code> object is a dictionary-like object with the following attributes:
    <br><br>
    * <code>data</code>, which is a pandas object (since <code>as_frame=True</code>). Each row corresponds to 8 feature values.
    <br><br>
    * <code>target</code> value contains average house value in units of 100,000. This is also a pandas object (since <code>as_frame=True</code>).
    <br><br>
    * <code>feature_names</code> is an array of ordered feature names used in the dataset.
    <br><br>
    * <code>DESCR</code> contains a description of the dataset.
    <br><br>
    * <code>frame</code> contains a dataframe with data and target.
    <br><br>
    Each of these attributes can be accessed as <code>&lt;bunch_object&gt;.key</code>. In our case, we can access these features as follows:
    <br><br>
    * <code>california_housing.data</code> gives us access to contents of <code>data</code> key.
    <br><br>
    * <code>california_housing.target</code> gives us access to contents of <code>target</code> key.
    <br><br>
    * <code>california_housing.feature_names</code> gives us access to contents of <code>feature_names</code> key.
    <br><br>
    * <code>california_housing.DESCR</code> gives us access to contents of <code>DESCR</code> key.
    <br><br>
    * <code>california_housing.frame</code> gives us access to contents of <code>frame</code> key.
</p>
</div>


# Dataset exploration

In [ ]:
print(california_housing.DESCR)

<a id="key-statistics"></a>
<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>KEY STATISTICS</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    Note down the key statistics from this description:
    <br><br>
    * There are <strong>20640 examples</strong> in the dataset.
    <br><br>
    * There are <strong>8 numerical attributes</strong> per example.
    <br><br>
    * The target label is the median house value.
    <br><br>
    * There are <strong>no missing values</strong> in this dataset.
</p>
</div>

In [ ]:
california_housing.data.shape


In [ ]:
type(california_housing.data)

In [ ]:
california_housing.target.shape

In [ ]:
type(california_housing.target)

# Feature names  
**Let's find out names of the attributes.**

In [ ]:
california_housing.feature_names

<a id="attributes-description"></a>
<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>ATTRIBUTES AND THEIR DESCRIPTION</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    Understanding the attributes and their descriptions is crucial for analyzing the dataset:
    <br><br>
    * <strong>MedInc</strong> - Median income in the block.
    <br><br>
    * <strong>HouseAge</strong> - Median house age in the block.
    <br><br>
    * <strong>AveRooms</strong> - Average number of rooms.
    <br><br>
    * <strong>AveBedrms</strong> - Average number of bedrooms.
    <br><br>
    * <strong>Population</strong> - Block population.
    <br><br>
    * <strong>AveOccupancy</strong> - Average house occupancy.
    <br><br>
    * <strong>Latitude</strong> - Latitude of the house block.
    <br><br>
    * <strong>Longitude</strong> - Longitude of the house block.
</p>
</div>


In [ ]:
california_housing.frame.head()

In [ ]:
california_housing.data.head()

In [ ]:
california_housing.target.head()


<a id="dataset-observations"></a>
<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>DATASET OBSERVATIONS</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    We observe that:
    <br><br>
    <ul>
        <li>The dataset contains <strong>20,640 examples</strong> with 8 features.</li>
        <li>All features are numerical features encoded as floating point numbers.</li>
        <li>There are <strong>no missing values</strong> in any features - the <code>Non-Null</code> is equal to the number of examples in the training set.</li>
    </ul>
</p>
</div>

# Feature and target histograms

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
california_housing.frame.hist(figsize=(12, 10), bins=30, edgecolor="black")
plt.subplots_adjust(hspace=0.7, wspace=0.4)

<a id="histogram-observations"></a>
<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>HISTOGRAM OBSERVATIONS</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    Let's observe these histogram and note down our findings:
    <br><br>
    <ul>
        <li><strong>MedInc</strong> has a long tail distribution - salary of people is more or less normally distributed with a few folks getting a high salary.</li>
        <li><strong>HouseAge</strong> has more or less a uniform distribution.</li>
        <li>The range for features, <strong>AveRooms</strong>, <strong>AveBedrms</strong>, <strong>AveOccupancy</strong>, <strong>Population</strong>, is large and it contains a small number of large values (as there are unnoticeable bins on the right in the histogram plots of these features). That would mean that there could be certain outlier values present in these features.</li>
        <li><strong>Latitude</strong> and <strong>Longitude</strong> carry geographical information. Their combination helps us decide the price of the house.</li>
        <li><strong>MedHouseVal</strong> also has a long tail distribution. It spikes towards the end. The reason is that the houses with a price more than 5 are given a value of 5.</li>
    </ul>
</p>
</div>


In [ ]:
california_housing.frame.describe()

<a id="outlier-observations"></a>
<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>OUTLIER OBSERVATIONS</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    We can observe that there is a large difference between <code>75%</code> and <code>max</code> values of <strong>AveRooms</strong>, <strong>AveBedrms</strong>, <strong>Population</strong>, and <strong>AveOccupancy</strong> - which confirms our intuition about the presence of outliers or extreme values in these features.
</p>
</div>

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


<a id="imports-regression"></a>
<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>IMPORTS FOR REGRESSION</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    For tackling regression problems, we need to import classes and utilities from the <code>sklearn.linear_model</code> module. This module provides implementations for various regression models, including:
    <ul>
        <li><code>LinearRegression</code></li>
        <li><code>SGDRegressor</code></li>
        <li><code>Ridge</code></li>
        <li><code>Lasso</code></li>
        <li><code>RidgeCV</code></li>
        <li><code>LassoCV</code></li>
    </ul>
    <br>
    Additionally, we need to bring in model selection utilities from the <code>sklearn.model_selection</code> module and metrics from the <code>sklearn.metrics</code> module.
    <br><br>
    Data preprocessing utilities are imported from the <code>sklearn.preprocessing</code> module.
</p>
</div>


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import loguniform
from scipy.stats import uniform

from sklearn.datasets import fetch_california_housing
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import Ridge
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import validation_curve
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [ ]:
np.random.seed(306)

**Let's use `ShuffleSplit` as cv with 10 splits and 20% examples set aside as test examples.**

In [ ]:
cv = ShuffleSplit(n_splits=10, test_size=0.2, random_state=42)

<a id="data-loading-splitting"></a>
<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>DATA LOADING AND SPLITTING</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    The data will be split into three parts:
    <ul>
        <li><strong>Train + Dev</strong>: Used for cross-validation to fine-tune model parameters.</li>
        <li><strong>Test</strong>: Reserved for evaluating the performance of the trained models.</li>
    </ul>
</p>
</div>


In [ ]:
# fetch dataset
features, labels = fetch_california_housing(as_frame=True, return_X_y=True)

# train-test split
com_train_features, test_features, com_train_labels, test_labels = train_test_split(
    features, labels, random_state=42)

# train --> train + dev split
train_features, dev_features, train_labels, dev_labels = train_test_split(
    com_train_features, com_train_labels, random_state=42)

<a id="linear-regression-normal-equation"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>LINEAR REGRESSION WITH NORMAL EQUATION</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    In this section, we will utilize the normal equation method to train a linear regression model. This approach allows us to find the optimal coefficients without iterative methods.
    <br><br>
    The pipeline is set up with two key stages:
    <ul>
        <li><strong>Feature Scaling</strong>: To scale the features appropriately.</li>
        <li><strong>Linear Regression</strong>: To perform regression on the transformed feature matrix.</li>
    </ul>
    <br>
    Throughout this notebook, we will follow a consistent pattern for each estimator:
    <ul>
        <li><strong>Pipeline</strong>: Combines data preprocessing and modeling steps.</li>
        <li><strong>cross_validate</strong>: Trains the model using <code>ShuffleSplit</code> cross-validation with <code>neg_mean_absolute_error</code> as the scoring metric.</li>
        <li><strong>Error Reporting</strong>: Converts scores to errors and reports mean absolute errors on the development set.</li>
    </ul>
</p>
</div>

In [ ]:
lin_reg_pipeline = Pipeline([("feature_scaling", StandardScaler()),
                             ("lin_reg", LinearRegression())])
lin_reg_cv_results = cross_validate(lin_reg_pipeline,
                                    com_train_features,
                                    com_train_labels,
                                    cv=cv,
                                    scoring="neg_mean_absolute_error",
                                    return_train_score=True,
                                    return_estimator=True)

lin_reg_train_error = -1 * lin_reg_cv_results['train_score']
lin_reg_test_error = -1 * lin_reg_cv_results['test_score']

print(f"Mean absolute error of linear regression model on the train set:\n"
      f"{lin_reg_train_error.mean():.3f} +/- {lin_reg_train_error.std():.3f}")
print(f"Mean absolute error of linear regression model on the test set:\n"
      f"{lin_reg_test_error.mean():.3f} +/- {lin_reg_test_error.std():.3f}")

**Both the errors are close, but are not low.  This points to underfitting.  We can address it by adding more feature through polynomial regression.**

<a id="linear-regression-sgd"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>LINEAR REGRESSION WITH SGD</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    In this section, we will employ the Stochastic Gradient Descent (SGD) method to train a linear regression model. This iterative optimization technique is particularly useful when dealing with large datasets.
    <br><br>
    The pipeline is configured with two essential stages:
    <ul>
        <li><strong>Feature Scaling</strong>: To standardize the features.</li>
        <li><strong>SGD Regression</strong>: To perform regression using the transformed feature matrix.</li>
    </ul>
</p>
</div>

In [ ]:
sgd_reg_pipeline = Pipeline([("feature_scaling", StandardScaler()),
                             ("sgd_reg", SGDRegressor(
                                 max_iter=int(np.ceil(1e6/com_train_features.shape[0])),
                                 early_stopping=True,
                                 eta0=1e-4,
                                 learning_rate='constant',
                                 tol=1e-5,
                                 validation_fraction=0.1,
                                 n_iter_no_change=5,
                                 average=10,
                                 random_state=42))])

sgd_reg_cv_results = cross_validate(sgd_reg_pipeline,
                                    com_train_features,
                                    com_train_labels,
                                    cv=cv,
                                    scoring="neg_mean_absolute_error",
                                    return_train_score=True,
                                    return_estimator=True)

sgd_train_error = -1 * sgd_reg_cv_results['train_score']
sgd_test_error = -1 * sgd_reg_cv_results['test_score']

print(f"Mean absolute error of SGD regression model on the train set:\n"
      f"{sgd_train_error.mean():.3f} +/- {sgd_train_error.std():.3f}")
print(f"Mean absolute error of SGD regression model on the test set:\n"
      f"{sgd_test_error.mean():.3f} +/- {sgd_test_error.std():.3f}")

<a id="polynomial-regression"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>POLYNOMIAL REGRESSION</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    In this segment, we will explore polynomial regression by training a model with a degree of 2. Additionally, we will use the <code>validation_curve</code> to determine the optimal degree for polynomial models.
    <br><br>
    The <code>PolynomialFeatures</code> function is employed to transform the features to the specified degree (in this case, 2). After transforming the features, we apply feature scaling before training the regression model.
</p>
</div>

In [ ]:
poly_reg_pipeline = Pipeline([("poly", PolynomialFeatures(degree=2)),
                              ("feature_scaling", StandardScaler()),
                              ("lin_reg", LinearRegression())])
poly_reg_cv_results = cross_validate(poly_reg_pipeline,
                                    com_train_features,
                                    com_train_labels,
                                    cv=cv,
                                    scoring="neg_mean_absolute_error",
                                    return_train_score=True,
                                    return_estimator=True)

poly_reg_train_error = -1 * poly_reg_cv_results['train_score']
poly_reg_test_error = -1 * poly_reg_cv_results['test_score']

print(f"Mean absolute error of linear regression model on the train set:\n"
      f"{poly_reg_train_error.mean():.3f} +/- {poly_reg_train_error.std():.3f}")
print(f"Mean absolute error of linear regression model on the test set:\n"
      f"{poly_reg_test_error.mean():.3f} +/- {poly_reg_test_error.std():.3f}")

<a id="interaction-features-in-polynomial-regression"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>INTERACTION FEATURES IN POLYNOMIAL REGRESSION</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    Upon examining the results, we observe that both the training and validation errors have decreased after utilizing second-order polynomial features in the model.
    <br><br>
    To optimize further, instead of employing all polynomial features, we focus on using only interaction feature terms in the polynomial model. This approach allows us to train a more efficient linear regression model.
</p>
</div>

In [ ]:
poly_reg_pipeline = Pipeline([("poly", PolynomialFeatures(degree=2, interaction_only=True)),
                              ("feature_scaling", StandardScaler()),
                              ("lin_reg", LinearRegression())])
poly_reg_cv_results = cross_validate(poly_reg_pipeline,
                                    com_train_features,
                                    com_train_labels,
                                    cv=cv,
                                    scoring="neg_mean_absolute_error",
                                    return_train_score=True,
                                    return_estimator=True)

poly_reg_train_error = -1 * poly_reg_cv_results['train_score']
poly_reg_test_error = -1 * poly_reg_cv_results['test_score']

print(f"Mean absolute error of polynomial regression model of degree 2 on the train set:\n"
      f"{poly_reg_train_error.mean():.3f} +/- {poly_reg_train_error.std():.3f}")
print(f"Mean absolute error of polynomial regression model of degree 2 on the test set:\n"
      f"{poly_reg_test_error.mean():.3f} +/- {poly_reg_test_error.std():.3f}")

<a id="choosing-polynomial-degree"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>CHOOSING THE RIGHT POLYNOMIAL DEGREE</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    To determine the most suitable polynomial degree for our regression problem, we will employ the <strong>validation_curve</strong>. This technique acts as a manual hyperparameter tuning method.
    <br><br>
    We will specify a list of polynomial degree values to explore, which will be passed as a parameter to the <strong>validation_curve</strong>. This allows us to identify the optimal degree that minimizes the error and improves the model's performance.
</p>
</div>

In [ ]:
degree = [1, 2, 3, 4, 5]
train_scores, test_scores = validation_curve(
    poly_reg_pipeline, com_train_features, com_train_labels, param_name="poly__degree",
    param_range=degree, cv=cv, scoring="neg_mean_absolute_error",
    n_jobs=2)

train_errors, test_errors = -train_scores, -test_scores
plt.plot(degree, train_errors.mean(axis=1), 'b-x', label="Training error")
plt.plot(degree, test_errors.mean(axis=1), 'r-x', label="Test error")
plt.legend()

plt.xlabel("degree")
plt.ylabel("Mean absolute error (k$)")
_ = plt.title("Validation curve for polynomial regression")

<a id="selecting-optimal-degree"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>SELECTING THE OPTIMAL POLYNOMIAL DEGREE</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    To identify the most effective polynomial degree, we focus on the degree that results in the lowest mean absolute error. In our analysis, a degree of <strong>2</strong> emerges as the optimal choice, as it yields the minimum mean absolute error. This degree will be selected for our polynomial regression model, ensuring a balance between model complexity and predictive accuracy.
</p>
</div>

<a id="ridge-regression"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>RIDGE REGRESSION</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    Polynomial models, particularly those with higher-order features, are prone to overfitting. To mitigate this, we utilize Ridge regression, which incorporates a regularization term to penalize excessive model complexity. For this analysis, we set the regularization rate <code>alpha</code> to <strong>0.5</strong> and train the regression model accordingly.
    <br><br>
    Subsequently, we will perform a hyperparameter search to determine the optimal value of <code>alpha</code> that minimizes cross-validation errors, ensuring an effective balance between model complexity and generalization.
</p>
</div>

In [ ]:
ridge_reg_pipeline = Pipeline([("poly", PolynomialFeatures(degree=2)),
                              ("feature_scaling", StandardScaler()),
                              ("ridge", Ridge(alpha=0.5))])
ridge_reg_cv_results = cross_validate(ridge_reg_pipeline,
                                    com_train_features,
                                    com_train_labels,
                                    cv=cv,
                                    scoring="neg_mean_absolute_error",
                                    return_train_score=True,
                                    return_estimator=True)

ridge_reg_train_error = -1 * ridge_reg_cv_results['train_score']
ridge_reg_test_error = -1 * ridge_reg_cv_results['test_score']

print(f"Mean absolute error of ridge regression model (alpha=0.5) on the train set:\n"
      f"{ridge_reg_train_error.mean():.3f} +/- {ridge_reg_train_error.std():.3f}")
print(f"Mean absolute error of ridge regression model (alpha=0.5) on the test set:\n"
      f"{ridge_reg_test_error.mean():.3f} +/- {ridge_reg_test_error.std():.3f}")

<a id="hpt-ridge-regularization-rate"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>HYPERPARAMETER TUNING FOR RIDGE REGULARIZATION RATE</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    To identify the optimal regularization rate <code>alpha</code> for Ridge regression, we perform hyperparameter tuning. This process involves evaluating a range of <code>alpha</code> values to determine which one results in the lowest cross-validation errors.
    <br><br>
    We will systematically test different values of <code>alpha</code> and use cross-validation to assess the model's performance for each value. The optimal <code>alpha</code> is selected based on which value yields the best balance between model fit and generalization, minimizing cross-validation errors.
</p>
</div>

In [ ]:
alpha_list = np.logspace(-4, 0, num=20)
ridge_reg_pipeline = Pipeline([("poly", PolynomialFeatures(degree=2)),
                              ("feature_scaling", StandardScaler()),
                              ("ridge_cv", RidgeCV(alphas=alpha_list,
                                                   cv=cv,
                                                   scoring="neg_mean_absolute_error"))])
ridge_reg_cv_results = ridge_reg_pipeline.fit(com_train_features, com_train_labels)

In [ ]:
print ("The score with the best alpha is:",
       f"{ridge_reg_cv_results[-1].best_score_:.3f}")
print ("The error with the best alpha is:",
       f"{-ridge_reg_cv_results[-1].best_score_:.3f}")

In [ ]:
print ("The best value for alpha:", ridge_reg_cv_results[-1].alpha_)

<a id="ridgecv-with-cross-validation"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>RIDGECV WITH CROSS VALIDATION</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    To identify the optimal regularization rate <code>alpha</code> for Ridge regression, we use <code>RidgeCV</code>, which performs cross-validation to select the best <code>alpha</code> from a specified list of candidate values.
    <br><br>
    We will provide a range of <code>alpha</code> values to <code>RidgeCV</code> and allow it to evaluate each value's performance through cross-validation. The best <code>alpha</code> is chosen based on which one minimizes cross-validation errors, ensuring that our model achieves the best balance between fit and generalization.
</p>
</div>

In [ ]:
alpha_list = np.logspace(-4, 0, num=20)
ridge_reg_pipeline = Pipeline([("poly", PolynomialFeatures(degree=2)),
                              ("feature_scaling", StandardScaler()),
                              ("ridge_cv", RidgeCV(alphas=alpha_list,
                                                   store_cv_values=True))])
ridge_reg_cv_results = cross_validate(ridge_reg_pipeline,
                                    com_train_features,
                                    com_train_labels,
                                    cv=cv,
                                    scoring="neg_mean_absolute_error",
                                    return_train_score=True,
                                    return_estimator=True)

ridge_reg_train_error = -1 * ridge_reg_cv_results['train_score']
ridge_reg_test_error = -1 * ridge_reg_cv_results['test_score']

print(f"Mean absolute error of ridge regression model on the train set:\n"
      f"{ridge_reg_train_error.mean():.3f} +/- {ridge_reg_train_error.std():.3f}")
print(f"Mean absolute error of ridge regression model on the test set:\n"
      f"{ridge_reg_test_error.mean():.3f} +/- {ridge_reg_test_error.std():.3f}")

**Let's look at the mean of mean absolute errors at different values of regularization rate across different cross validation folds.**

In [ ]:
mse_alphas = [est[-1].cv_values_.mean(axis=0)
              for est in ridge_reg_cv_results["estimator"]]
cv_alphas = pd.DataFrame(mse_alphas, columns=alpha_list)
cv_alphas

In [ ]:
cv_alphas.mean(axis=0).plot(marker="+")
plt.ylabel("Mean squared error\n (lower is better)")
plt.xlabel("alpha")
_ = plt.title("Error obtained by cross-validation")

In [ ]:
best_alphas = [est[-1].alpha_ for est in ridge_reg_cv_results["estimator"]]
best_alphas

<a id="ridgecv-final-estimate"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>RIDGECV FINAL ESTIMATE</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    The optimal regularization strength for Ridge regression, as determined by <code>RidgeCV</code>, may vary across different cross-validation iterations. 
    <br><br>
    However, it is standard practice to average the best <code>alpha</code> values found across multiple cross-validation folds. This approach provides a robust final estimate for the regularization parameter, reflecting the overall performance of the model across different data samples.
</p>
</div>

In [ ]:
print(f"The mean optimal alpha leading to the best generalization performance is:\n"
      f"{np.mean(best_alphas):.2f} +/- {np.std(best_alphas):.2f}")


<a id="ridge-hpt-gridsearchcv"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>RIDGE HPT THROUGH `GRIDSEARCHCV`</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    To identify the optimal regularization strength for Ridge regression, we can use <code>GridSearchCV</code> for hyperparameter tuning. 
    <br><br>

In [ ]:
ridge_grid_pipeline = Pipeline([("poly", PolynomialFeatures(degree=2)),
                              ("feature_scaling", StandardScaler()),
                              ("ridge", Ridge())])

param_grid = {'poly__degree': (1, 2, 3),
              'ridge__alpha': np.logspace(-4, 0, num=20)}
ridge_grid_search = GridSearchCV(ridge_grid_pipeline,
                                 param_grid=param_grid,
                                 n_jobs=2,
                                 cv=cv,
                                 scoring="neg_mean_absolute_error",
                                 return_train_score=True)
ridge_grid_search.fit(com_train_features, com_train_labels)

**`ridge_grid_search.best_index_` gives us the index of the best parameter in the list**

In [ ]:
mean_train_error = -1*ridge_grid_search.cv_results_['mean_train_score'][ridge_grid_search.best_index_]
mean_test_error = -1*ridge_grid_search.cv_results_['mean_test_score'][ridge_grid_search.best_index_]
std_train_error = ridge_grid_search.cv_results_['std_train_score'][ridge_grid_search.best_index_]
std_test_error = ridge_grid_search.cv_results_['std_train_score'][ridge_grid_search.best_index_]

print(f"Best Mean absolute error of polynomial ridge regression model on the train set:\n"
      f"{mean_train_error:.3f} +/- {std_train_error:.3f}")
print(f"Mean absolute error of polynomial ridge regression model on the test set:\n"
      f"{mean_test_error:.3f} +/- {std_test_error:.3f}")

In [ ]:
print ("Mean cross validated score of the best estimator is: ", ridge_grid_search.best_score_)
print ("Mean cross validated error of the best estimator is: ", -ridge_grid_search.best_score_)

**Note that this is same as RidgeCV that we carried out earlier.**

In [ ]:
print ("The best parameter value is:", ridge_grid_search.best_params_)

<a id="lasso-regression-baseline"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>LASSON REGRESSION: BASELINE MODEL WITH FIXED LEARNING RATE</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    In this section, we will explore Lasso regression by setting up a baseline model with a fixed learning rate.

In [ ]:
lasso_reg_pipeline = Pipeline([("poly", PolynomialFeatures(degree=2)),
                              ("feature_scaling", StandardScaler()),
                              ("lasso", Lasso(alpha=0.01))])
lasso_reg_cv_results = cross_validate(lasso_reg_pipeline,
                                    com_train_features,
                                    com_train_labels,
                                    cv=cv,
                                    scoring="neg_mean_absolute_error",
                                    return_train_score=True,
                                    return_estimator=True)

lasso_reg_train_error = -1 * lasso_reg_cv_results['train_score']
lasso_reg_test_error = -1 * lasso_reg_cv_results['test_score']

print(f"Mean absolute error of linear regression model on the train set:\n"
      f"{lasso_reg_train_error.mean():.3f} +/- {ridge_reg_train_error.std():.3f}")
print(f"Mean absolute error of linear regression model on the test set:\n"
      f"{lasso_reg_test_error.mean():.3f} +/- {ridge_reg_test_error.std():.3f}")

<a id="hpt-lasso-regularization-rate"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>HYPERPARAMETER TUNING FOR LASSO REGULARIZATION RATE WITH CROSS VALIDATION</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    In this section, we will perform hyperparameter tuning for the Lasso regression model to find the optimal regularization rate using cross-validation.

In [ ]:
alpha_list = np.logspace(-6, 0, num=20)
lasso_reg_pipeline = Pipeline([("poly", PolynomialFeatures(degree=2)),
                              ("feature_scaling", StandardScaler()),
                              ("lasso_cv", LassoCV(alphas=alpha_list, cv=cv))])
lasso_reg_cv_results = cross_validate(lasso_reg_pipeline,
                                    com_train_features,
                                    com_train_labels,
                                    cv=cv,
                                    scoring="neg_mean_absolute_error",
                                    return_train_score=True,
                                    return_estimator=True)

lasso_reg_train_error = -1 * lasso_reg_cv_results['train_score']
lasso_reg_test_error = -1 * lasso_reg_cv_results['test_score']

print(f"Mean absolute error of linear regression model on the train set:\n"
      f"{lasso_reg_train_error.mean():.3f} +/- {lasso_reg_train_error.std():.3f}")
print(f"Mean absolute error of linear regression model on the test set:\n"
      f"{lasso_reg_test_error.mean():.3f} +/- {lasso_reg_test_error.std():.3f}")

In [ ]:
best_alphas = [est[-1].alpha_ for est in lasso_reg_cv_results["estimator"]]
best_alphas

In [ ]:
print(f"The mean optimal alpha leading to the best generalization performance is:\n"
      f"{np.mean(best_alphas):.2f} +/- {np.std(best_alphas):.2f}")

In [ ]:
lasso_reg_pipeline = Pipeline([("poly", PolynomialFeatures(degree=2)),
                              ("feature_scaling", StandardScaler()),
                              ("lasso", Lasso(alpha=0.01))])
lasso_reg_pipeline.fit(com_train_features, com_train_labels)
train_error = mean_absolute_error(com_train_labels,
                                 lasso_reg_pipeline.predict(com_train_features))

print(f"Mean absolute error of Lasso CV model on the train set:", train_error)

**with GridSearchCV**

In [ ]:
lasso_grid_pipeline = Pipeline([("poly", PolynomialFeatures()),
                              ("feature_scaling", StandardScaler()),
                              ("lasso", Lasso())])

param_grid = {'poly__degree': (1, 2, 3),
              'lasso__alpha': np.logspace(-4, 0, num=20)}
lasso_grid_search = GridSearchCV(lasso_grid_pipeline,
                                 param_grid=param_grid,
                                 n_jobs=2,
                                 cv=cv,
                                 scoring="neg_mean_absolute_error",
                                 return_train_score=True)
lasso_grid_search.fit(com_train_features, com_train_labels)

In [ ]:
mean_train_error = -1*lasso_grid_search.cv_results_['mean_train_score'][lasso_grid_search.best_index_]
mean_test_error = -1*lasso_grid_search.cv_results_['mean_test_score'][lasso_grid_search.best_index_]
std_train_error = lasso_grid_search.cv_results_['std_train_score'][lasso_grid_search.best_index_]
std_test_error = lasso_grid_search.cv_results_['std_train_score'][lasso_grid_search.best_index_]

print(f"Best Mean absolute error of polynomial ridge regression model on the train set:\n"
      f"{mean_train_error:.3f} +/- {std_train_error:.3f}")
print(f"Mean absolute error of polynomial ridge regression model on the test set:\n"
      f"{mean_test_error:.3f} +/- {std_test_error:.3f}")

In [ ]:
print ("Mean cross validated score of the best estimator is: ", lasso_grid_search.best_score_)

In [ ]:
print ("The best parameter value is:", lasso_grid_search.best_params_)

<a id="sgd-regularization-hpt"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>SGD REGRESSION: REGULARIZATION AND HYPERPARAMETER TUNING</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    Stochastic Gradient Descent (SGD) is another method for regularization in linear regression. Unlike standard Linear Regression, SGDRegressor includes additional hyperparameters that need careful tuning to match or exceed the performance of other regression models.

In [ ]:
poly_sgd_pipeline = Pipeline([("poly", PolynomialFeatures()),
                              ("feature_scaling", StandardScaler()),
                              ("sgd_reg", SGDRegressor(
                                 penalty='elasticnet',
                                 random_state=42))])
poly_sgd_cv_results = cross_validate(poly_sgd_pipeline,
                                    com_train_features,
                                    com_train_labels,
                                    cv=cv,
                                    scoring="neg_mean_absolute_error",
                                    return_train_score=True,
                                    return_estimator=True)

poly_sgd_train_error = -1 * poly_sgd_cv_results['train_score']
poly_sgd_test_error = -1 * poly_sgd_cv_results['test_score']

print(f"Mean absolute error of linear regression model on the train set:\n"
      f"{poly_sgd_train_error.mean():.3f} +/- {poly_sgd_train_error.std():.3f}")
print(f"Mean absolute error of linear regression model on the test set:\n"
      f"{poly_sgd_test_error.mean():.3f} +/- {poly_sgd_test_error.std():.3f}")

<a id="search-best-parameters-randomizedsearchcv"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>SEARCHING FOR BEST PARAMETERS WITH RANDOMIZEDSEARCHCV</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    To find the optimal set of parameters for the Polynomial + SGD pipeline, we will use <code>RandomizedSearchCV</code>. This technique allows us to efficiently search through a range of hyperparameters to identify the best combination for our model.

In [ ]:
class uniform_int:
    """Integer valued version of the uniform distribution"""
    def __init__(self, a, b):
        self._distribution = uniform(a, b)

    def rvs(self, *args, **kwargs):
        """Random variable sample"""
        return self._distribution.rvs(*args, **kwargs).astype(int)

**Let's specify RandomizedSearchCV set up.**

In [ ]:
param_distributions = {
    'poly__degree': [1, 2, 3],
    'sgd_reg__learning_rate': ['constant', 'adaptive', 'invscaling'],
    'sgd_reg__l1_ratio': uniform(0, 1),
    'sgd_reg__eta0': loguniform(1e-5, 1),
    'sgd_reg__power_t': uniform(0, 1)
}

poly_sgd_random_search = RandomizedSearchCV(
    poly_sgd_pipeline, param_distributions=param_distributions,
    n_iter=10, cv=cv, verbose=1, scoring='neg_mean_absolute_error'
)
poly_sgd_random_search.fit(com_train_features, com_train_labels)

In [ ]:
poly_sgd_random_search.best_score_

In [ ]:
poly_sgd_random_search.best_params_

**And the best estimator can be accessed with best_estimator_ member variable.**

<a id="comparison-of-weight-vectors"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>COMPARISON OF WEIGHT VECTORS</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    In this section, we will compare the weight vectors produced by different regression models. The weight vectors provide insights into the importance of each feature in predicting the target variable.

In [ ]:
feature_names = poly_reg_cv_results["estimator"][0][0].get_feature_names_out(
    input_features=train_features.columns)
feature_names

In [ ]:
coefs = [est[-1].coef_ for est in poly_reg_cv_results["estimator"]]
weights_polynomial_regression = pd.DataFrame(coefs, columns=feature_names)

In [ ]:
color = {"whiskers": "black", "medians": "black", "caps": "black"}
weights_polynomial_regression.plot.box(color=color, vert=False, figsize=(6, 16))
_ = plt.title("Polynomial regression coefficients")

In [ ]:
feature_names = ridge_reg_cv_results["estimator"][0][0].get_feature_names_out(
    input_features=train_features.columns)
feature_names

In [ ]:
coefs = [est[-1].coef_ for est in ridge_reg_cv_results["estimator"]]
weights_ridge_regression = pd.DataFrame(coefs, columns=feature_names)

In [ ]:
color = {"whiskers": "black", "medians": "black", "caps": "black"}
weights_ridge_regression.plot.box(color=color, vert=False, figsize=(6, 16))
_ = plt.title("Polynomial regression coefficients")

<a id="performance-on-the-test-set"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>PERFORMANCE ON THE TEST SET</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">

<a id="baseline"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>BASELINE</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">

In [ ]:
baseline_model_median = DummyRegressor(strategy='median')
baseline_model_median.fit(train_features, train_labels)
mean_absolute_percentage_error(test_labels,
                               baseline_model_median.predict(test_features))


<a id="linear-regression-with-normal-equation"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>LINEAR REGRESSION WITH NORMAL EQUATION</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">

In [ ]:
mean_absolute_percentage_error(test_labels,
                               lin_reg_cv_results['estimator'][0].predict(
                                   test_features))

In [ ]:
mean_absolute_percentage_error(test_labels,
                               poly_sgd_random_search.best_estimator_.predict(
                                   test_features))


<a id="polynomial-regression"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>POLYNOMIAL REGRESSION</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">

In [ ]:
poly_reg_pipeline.fit(com_train_features, com_train_labels)
mean_absolute_percentage_error(test_labels,
                               poly_reg_pipeline.predict(test_features))

<a id="ridge-regression"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>RIDGE REGRESSION</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">

In [ ]:
mean_absolute_percentage_error(test_labels,
                               ridge_grid_search.best_estimator_.predict(test_features))

<a id="lasso-regression"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>LASSO REGRESSION</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    We will retrain the Lasso regression model using the optimal <code>alpha</code> value identified through hyperparameter tuning. After retraining the model, we will evaluate its performance on the test data to assess its effectiveness.
</p>
</div>

In [ ]:
mean_absolute_percentage_error(test_labels,
                               lasso_grid_search.best_estimator_.predict(test_features))

<a id="summary"></a>

<div style="background-color: #e3f2fd; font-size:150%; text-align:left; border: 7px solid #0288d1; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); font-family: Calibri; border-radius: 20px; padding: 10px; width:95%">
<h1 align="center"><font color=#0277bd><strong>SUMMARY</strong></font></h1>
<p style="font-family: Calibri; color: #0277bd; text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.3); text-align: left; font-size: 110%">
    We trained multiple linear regression models on the housing dataset, optimizing their hyperparameters through systematic hyperparameter tuning. After identifying the best hyperparameters, we retrained the models and evaluated their performance on a test set that was reserved for final assessment.
    <br><br>
    This approach mirrors how real-world problems are typically addressed, starting with simple models and progressively advancing to more sophisticated techniques as needed.
</p>
</div>